# Visualizing Hi-C Data with cooler and cooltools

This notebook demonstrates how to visualize contact matrices from the `.mcool` files we converted previously. We'll cover plotting a full chromosome, zooming into a specific region, and comparing different normalizations and colormaps.

Reference: Cooltools Viz Documentation

In [ ]:
import cooler
import cooltools
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# cooltools registers the 'fall' colormap, which is standard for Hi-C
import cooltools.lib.plotting

## 1. Load the data

An `.mcool` file contains multiple resolutions in a single file. Let's load the 1Mb (1,000,000 bp) resolution for whole-chromosome visualization, and the 100kb (100,000 bp) resolution for zoomed-in views.

In [ ]:
# Load cooler objects
clr_1mb = cooler.Cooler('data/Control_inter_30.mcool::/resolutions/1000000')
clr_100kb = cooler.Cooler('data/Control_inter_30.mcool::/resolutions/100000')

print("Chromosomes available:", clr_1mb.chromnames[:5])

## 2. Visualize a full chromosome

We'll fetch the contact matrix for the first chromosome in the list and plot it. Because Hi-C interaction frequencies span several orders of magnitude, we use a logarithmic color scale (`LogNorm`).

In [ ]:
chrom = clr_1mb.chromnames[0]
matrix_full = clr_1mb.matrix(balance=False).fetch(chrom)

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.matshow(matrix_full, norm=LogNorm(vmin=1, vmax=1000), cmap='fall')
plt.colorbar(im, label='Raw Contact Frequency', fraction=0.046, pad=0.04)
ax.set_title(f'Full Chromosome: {chrom} (1 Mb resolution)', y=1.05)
plt.show()

## 3. Zoomed region

Let's zoom into a 20-megabase region on the same chromosome. We will use the `clr_100kb` cooler object so the bins have enough detail.

In [ ]:
start, end = 10_000_000, 30_000_000
region = f'{chrom}:{start}-{end}'

matrix_zoom = clr_100kb.matrix(balance=False).fetch(region)

fig, ax = plt.subplots(figsize=(8, 8))
# We use a different colormap ('Reds') here for variety
im = ax.matshow(matrix_zoom, norm=LogNorm(vmin=1, vmax=200), cmap='Reds')
plt.colorbar(im, label='Raw Contact Frequency', fraction=0.046, pad=0.04)
ax.set_title(f'Zoomed Region: {region} (100 kb resolution)', y=1.05)
plt.show()

## 4. Normalizations and Color Comparisons

Hi-C data is almost always iteratively balanced (using ICE, KR, or VC methods) to remove experimental biases. 

Let's compare raw counts with balanced counts side-by-side using the `balance=True` flag in cooler.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: Raw counts
matrix_raw = clr_100kb.matrix(balance=False).fetch(region)
im0 = axs[0].matshow(matrix_raw, norm=LogNorm(vmin=1, vmax=200), cmap='Reds')
axs[0].set_title('Raw Counts', y=1.05)
plt.colorbar(im0, ax=axs[0], shrink=0.8, label='Raw Contacts')

# Plot 2: Balanced counts
try:
    matrix_bal = clr_100kb.matrix(balance=True).fetch(region)
    # Balanced matrices have much smaller values (they represent probabilities), so we adjust vmin/vmax accordingly
    im1 = axs[1].matshow(matrix_bal, norm=LogNorm(vmin=1e-4, vmax=5e-3), cmap='fall')
    axs[1].set_title('Balanced (ICE / KR)', y=1.05)
    plt.colorbar(im1, ax=axs[1], shrink=0.8, label='Balanced Frequencies')
except ValueError:
    # Fallback if the cool file doesn't contain balancing weights
    axs[1].text(0.5, 0.5, "Balancing weights not found in this file", 
                ha='center', va='center', fontsize=12)
    axs[1].set_title('Balanced - Not Available', y=1.05)

plt.tight_layout()
plt.show()